In [85]:
import requests, time, pandas as pd

GQL_URL = "https://orchestrator.pgatour.com/graphql"
HEADERS = {"User-Agent":"Mozilla/5.0",
           "X-Api-Key":"da2-gsrx5bibzbb4njvhl7t37wqyl4",
           "Content-Type":"application/json"}

QUERY = """
query TournamentPastResults($tournamentPastResultsId: ID!, $year: Int) {
  tournamentPastResults(id: $tournamentPastResultsId, year: $year) {
    players {
      position
      total
      parRelativeScore
      player { id displayName }
      rounds { score parRelativeScore }
    }
  }
}
"""

def fetch_tournament_info_historical():
    all_tournaments = []
    for year in range(2010, 2026):
        payload = requests.get(f"https://data-api.pgatour.com/schedule/R/{year}?sort=ASC", timeout=30).json()
        for t in payload.get("tournaments", []):
            t["season_year"] = year
            all_tournaments.append(t)
    return pd.DataFrame(all_tournaments)

def fetch_tournament_results_historical():
    info_df = fetch_tournament_info_historical()

    name_map = (info_df[["tournamentId", "name"]]
                .dropna().drop_duplicates("tournamentId")
                .set_index("tournamentId")["name"].to_dict())

    tids = info_df["tournamentId"].dropna().drop_duplicates().astype(str).tolist()

    rows = []
    for tid in tids:
        yr_from_id = int(tid[1:5])
        year_param = int(f"{yr_from_id}0")   # 20250

        resp = requests.post(
        GQL_URL,
        headers=HEADERS,
        json={
            "query": QUERY,
            "operationName": "TournamentPastResults",
            "variables": {"tournamentPastResultsId": tid, "year": year_param},
        },
    ).json()

        if resp.get("errors"):
            print(f"ERROR: {tid} ({yr_from_id}) -> {resp['errors'][0].get('message')}")
            continue

        players = resp.get("data", {}).get("tournamentPastResults", {}).get("players", []) or []
        tname = name_map.get(tid)

        for p in players:
            player = p.get("player") or {}
            row = {
                "tournamentId": tid,
                "season_year": yr_from_id,
                "tournament_key": tname,
                "position": p.get("position"),
                "total": p.get("total"),
                "parRelativeScore": p.get("parRelativeScore"),
                "playerId": player.get("id"),
                "playerName": player.get("displayName"),
            }
            for i, rd in enumerate(p.get("rounds") or [], 1):
                row[f"r{i}_score"] = rd.get("score")
                row[f"r{i}_par"] = rd.get("parRelativeScore")
            rows.append(row)

        print(f"Done: {tid} ({yr_from_id}) {tname} players={len(players)}")
        time.sleep(0.15)

    return pd.DataFrame(rows)

df = fetch_tournament_results_historical()


Done: R2012016 (2012) Hyundai Tournament of Champions players=27
Done: R2012006 (2012) Sony Open in Hawaii players=144
Done: R2012002 (2012) Humana Challenge in partnership with the Clinton Foundation players=144
Done: R2012004 (2012) Farmers Insurance Open players=156
Done: R2012003 (2012) Waste Management Phoenix Open players=132
Done: R2012005 (2012) AT&T Pebble Beach National Pro-Am players=156
Done: R2012007 (2012) Northern Trust Open players=144
Done: R2012470 (2012) World Golf Championships-Accenture Match Play Championship players=64
Done: R2012457 (2012) Mayakoba Golf Classic players=132
Done: R2012010 (2012) The Honda Classic players=144
Done: R2012473 (2012) World Golf Championships-Cadillac Championship players=0
Done: R2012483 (2012) Puerto Rico Open presented by seepuertorico.com players=132
Done: R2012475 (2012) Transitions Championship players=144
Done: R2012485 (2012) Tavistock Cup players=0
Done: R2012009 (2012) Arnold Palmer Invitational presented by MasterCard playe

Done: R2014025 (2014) FedEx St. Jude Classic players=0
Done: R2014026 (2014) U.S. Open players=156
Done: R2014034 (2014) Travelers Championship players=156
Done: R2014486 (2014) CVS Caremark Charity Classic players=0
Done: R2014471 (2014) Quicken Loans National players=0
Done: R2014490 (2014) The Greenbrier Classic players=0
Done: R2014030 (2014) John Deere Classic players=155
Done: R2014100 (2014) The Open Championship players=156
Done: R2014032 (2014) RBC Canadian Open players=156
Done: R2014476 (2014) World Golf Championships-Bridgestone Invitational players=0
Done: R2014472 (2014) Barracuda Championship players=132
Done: R2014033 (2014) PGA Championship players=156
Done: R2014013 (2014) Wyndham Championship players=156
Done: R2014027 (2014) The Barclays players=122
Done: R2014505 (2014) Deutsche Bank Championship players=0
Done: R2014028 (2014) BMW Championship players=69
Done: R2014060 (2014) TOUR Championship by Coca-Cola players=29
Done: R2014468 (2014) Ryder Cup players=24
Done

Done: R2017475 (2017) Valspar Championship players=144
Done: R2017009 (2017) Arnold Palmer Invitational presented by Mastercard players=120
Done: R2017470 (2017) World Golf Championships-Dell Technologies Match Play players=64
Done: R2017483 (2017) Puerto Rico Open players=132
Done: R2017020 (2017) Shell Houston Open players=144
Done: R2017014 (2017) Masters Tournament players=93
Done: R2017012 (2017) RBC Heritage players=132
Done: R2017041 (2017) Valero Texas Open players=156
Done: R2017018 (2017) Zurich Classic of New Orleans players=0
Done: R2017480 (2017) Wells Fargo Championship players=156
Done: R2017011 (2017) THE PLAYERS Championship players=145
Done: R2017019 (2017) AT&T Byron Nelson players=156
Done: R2017021 (2017) DEAN & DELUCA Invitational players=121
Done: R2017023 (2017) the Memorial Tournament presented by Nationwide players=120
Done: R2017025 (2017) FedEx St. Jude Classic players=0
Done: R2017026 (2017) U.S. Open players=156
Done: R2017486 (2017) CVS Health Charity Cla

Done: R2020020 (2020) Houston Open players=144
Done: R2020521 (2020) THE CJ CUP @ NINE BRIDGES players=78
Done: R2020530 (2020) MGM Resorts The Challenge: Japan Skins players=0
Done: R2020527 (2020) ZOZO CHAMPIONSHIP players=78
Done: R2020489 (2020) World Golf Championships-HSBC Champions players=0
Done: R2020528 (2020) Bermuda Championship players=120
Done: R2020457 (2020) Mayakoba Golf Classic players=132
Done: R2020493 (2020) The RSM Classic players=156
Done: R2020478 (2020) Hero World Challenge players=18
Done: R2020500 (2020) Presidents Cup players=0
Done: R2020058 (2020) QBE Shootout players=24
Done: R2020016 (2020) Sentry Tournament of Champions players=34
Done: R2020006 (2020) Sony Open in Hawaii players=144
Done: R2020002 (2020) The American Express players=156
Done: R2020004 (2020) Farmers Insurance Open players=156
Done: R2020003 (2020) Waste Management Phoenix Open players=132
Done: R2020005 (2020) AT&T Pebble Beach Pro-Am players=156
Done: R2020007 (2020) The Genesis Invit

Done: R2023464 (2023) Fortinet Championship players=156
Done: R2023500 (2023) Presidents Cup players=0
Done: R2023054 (2023) Sanderson Farms Championship players=144
Done: R2023047 (2023) Shriners Children's Open players=144
Done: R2023527 (2023) ZOZO CHAMPIONSHIP players=78
Done: R2023521 (2023) THE CJ CUP in South Carolina players=78
Done: R2023528 (2023) Butterfield Bermuda Championship players=132
Done: R2023457 (2023) World Wide Technology Championship players=132
Done: R2023020 (2023) Cadence Bank Houston Open players=132
Done: R2023493 (2023) The RSM Classic players=155
Done: R2023478 (2023) Hero World Challenge players=20
Done: R2023058 (2023) QBE Shootout players=24
Done: R2023016 (2023) Sentry Tournament of Champions players=39
Done: R2023006 (2023) Sony Open in Hawaii players=144
Done: R2023002 (2023) The American Express players=156
Done: R2023004 (2023) Farmers Insurance Open players=156
Done: R2023005 (2023) AT&T Pebble Beach Pro-Am players=156
Done: R2023003 (2023) WM Ph

Done: R2025541 (2025) Genesis Scottish Open players=156
Done: R2025518 (2025) ISCO Championship players=156
Done: R2025100 (2025) The Open Championship players=156
Done: R2025472 (2025) Barracuda Championship players=156
Done: R2025525 (2025) 3M Open players=156
Done: R2025013 (2025) Wyndham Championship players=156
Done: R2025027 (2025) FedEx St. Jude Championship players=69
Done: R2025028 (2025) BMW Championship players=49
Done: R2025060 (2025) TOUR Championship players=30
Done: R2025464 (2025) Procore Championship players=144
Done: R2025468 (2025) Ryder Cup players=0
Done: R2025054 (2025) Sanderson Farms Championship players=132
Done: R2025527 (2025) Baycurrent Classic players=78
Done: R2025554 (2025) Bank of Utah Championship players=132
Done: R2025457 (2025) World Wide Technology Championship players=120
Done: R2025528 (2025) Butterfield Bermuda Championship players=120
Done: R2025493 (2025) The RSM Classic players=156
Done: R2025478 (2025) Hero World Challenge players=20
Done: R2

In [90]:
import pandas as pd

def tournament_course_history(df: pd.DataFrame, tournament_name: str) -> pd.DataFrame:
    """
    Pass your full results df + a tournament_key name (e.g. "Sony Open in Hawaii").
    Returns a player-level ranking dataframe with sensible handling of Txx / CUT / WD / DQ.
    """

    tdf = df[df["tournament_key"].eq(tournament_name)].copy()
    if tdf.empty:
        return pd.DataFrame()

    # --- Position parsing: "1"->1, "T58"->58, "CUT"/"WD"/"DQ"/"DNS"->NaN ---
    pos_raw = tdf["position"].astype(str).str.upper().str.strip()
    tdf["finish_pos"] = (
        pos_raw.str.replace(r"[^0-9]", "", regex=True)
        .replace("", pd.NA)
    )
    tdf["finish_pos"] = pd.to_numeric(tdf["finish_pos"], errors="coerce")

    # Made-cut flag (explicit, not "finish_pos <= 70")
    tdf["made_cut"] = ~pos_raw.isin(["CUT", "WD", "DQ", "DNS"])

    # Numeric coercions for scoring
    for c in ["total", "r1_score", "r2_score", "r3_score", "r4_score"]:
        if c in tdf.columns:
            tdf[c] = pd.to_numeric(tdf[c], errors="coerce")

    # Player historical aggregates
    player_hist = (
        tdf.groupby("playerName", dropna=False)
        .agg(
            starts=("position", "count"),                 # true appearances
            made_cuts=("made_cut", "sum"),
            cut_rate=("made_cut", "mean"),
            avg_finish=("finish_pos", "mean"),            # only where numeric
            best_finish=("finish_pos", "min"),
            avg_total=("total", "mean"),
            avg_r1=("r1_score", "mean"),
            avg_r2=("r2_score", "mean"),
            avg_r3=("r3_score", "mean"),
            avg_r4=("r4_score", "mean"),
        )
        .reset_index()
    )

    # Derived course-fit features
    player_hist["early_round_avg"] = player_hist[["avg_r1", "avg_r2"]].mean(axis=1)
    player_hist["sunday_delta"] = player_hist["avg_r4"] - player_hist["avg_r1"]

    # Course-fit score (tunable weights)
    player_hist["course_fit_score"] = (
        (100 - player_hist["avg_finish"]) * 0.45 +
        (player_hist["cut_rate"] * 100) * 0.25 +
        (70 - player_hist["early_round_avg"]) * 0.20 +
        (5 - player_hist["sunday_delta"]) * 0.10
    )

    # Recent form at this course: last 3 appearances (numeric finishes only)
    if "season_year" in tdf.columns:
        tdf["season_year"] = pd.to_numeric(tdf["season_year"], errors="coerce")
        tdf["year_rank"] = tdf.groupby("playerName")["season_year"].rank(ascending=False, method="first")
        recent = tdf[(tdf["year_rank"] <= 3) & (tdf["finish_pos"].notna())]
        recent_perf = recent.groupby("playerName")["finish_pos"].mean().rename("recent_avg_finish")
        player_hist = player_hist.merge(recent_perf, on="playerName", how="left")
        player_hist["final_score"] = (
            player_hist["course_fit_score"] * 0.7 +
            (100 - player_hist["recent_avg_finish"]) * 0.3
        )
        sort_col = "final_score"
    else:
        player_hist["final_score"] = player_hist["course_fit_score"]
        sort_col = "final_score"

    # Filter out tiny samples (optional but recommended)
    out = player_hist[player_hist["starts"] >= 2].sort_values(sort_col, ascending=False).reset_index(drop=True)
    return out


In [103]:
df['tournament_key'].unique()

array(['Hyundai Tournament of Champions', 'Sony Open in Hawaii',
       'Humana Challenge in partnership with the Clinton Foundation',
       'Farmers Insurance Open', 'Waste Management Phoenix Open',
       'AT&T Pebble Beach National Pro-Am', 'Northern Trust Open',
       'World Golf Championships-Accenture Match Play Championship',
       'Mayakoba Golf Classic', 'The Honda Classic',
       'Puerto Rico Open presented by seepuertorico.com',
       'Transitions Championship',
       'Arnold Palmer Invitational presented by MasterCard',
       'Shell Houston Open', 'Masters Tournament', 'RBC Heritage',
       'Valero Texas Open', 'Zurich Classic of New Orleans',
       'Wells Fargo Championship', 'THE PLAYERS Championship',
       'HP Byron Nelson Championship',
       'Crowne Plaza Invitational at Colonial',
       'the Memorial Tournament presented by Nationwide Insurance',
       'U.S. Open', 'Travelers Championship', 'John Deere Classic',
       'The Open Championship', 'True Sout

In [104]:
import re

def normalize_tournament_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)                 # collapse whitespace
    s = re.sub(r"\(\d{4}\)$", "", s).strip()   # remove trailing "(2020)"
    s = s.lower()
    return s


ALIAS = {
    # PLAYERS
    "the players championship": [
        "the players championship", "the players"
    ],

    # SENTRY TOC
    "sentry tournament of champions": [
        "hyundai tournament of champions", "sbs tournament of champions",
        "sentry tournament of champions", "the sentry", "the sentry tournament of champions"
    ],

    # GENESIS @ riviera
    "the genesis invitational": [
        "northern trust open", "genesis open", "the genesis invitational"
    ],

    # MEMORIAL
    "the memorial tournament": [
        "the memorial tournament presented by nationwide insurance",
        "the memorial tournament presented by nationwide",
        "the memorial tournament presented by workday",
        "the memorial tournament"
    ],

    # TOUR CHAMP
    "tour championship": [
        "tour championship", "tour championship by coca-cola", "tour championship by coca cola"
    ],

    # CJ CUP
    "the cj cup": [
        "the cj cup @ nine bridges", "the cj cup @ shadow creek",
        "the cj cup @ summit", "the cj cup in south carolina",
        "the cj cup byron nelson", "the cj cup"
    ],

    # FORTINET / SAFEWAY
    "fortinet championship": [
        "safeway open", "fortinet championship", "procore championship"
    ],
}

ALIAS_REV = {}
for canon, variants in ALIAS.items():
    for v in variants:
        ALIAS_REV[normalize_tournament_name(v)] = canon


df["tournament_norm"] = df["tournament_key"].apply(normalize_tournament_name)
df["tournament_canon"] = df["tournament_norm"].map(ALIAS_REV).fillna(df["tournament_norm"])


In [107]:
df.to_csv('/home/zburnside/pga_prediction_model/historical_tournament_results.csv')
